# 02. Visualizaciones

Este notebook contiene las 4 secciones del proyecto, con 4 salidas por seccion.

## Librerias

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 110


## Cargar base maestra y proveedores

In [ ]:
BASE_DIR = Path('..').resolve()
DATA_DIR = BASE_DIR / 'data'
df = pd.read_csv(DATA_DIR / 'contrataciones_peru_2022_2024_maestro.csv', low_memory=False)

proveedores = []
for anio in [2022, 2023, 2024]:
    temp = pd.read_csv(DATA_DIR / str(anio) / 'awards_suppliers.csv', usecols=['main_ocid', 'name'], low_memory=False)
    temp['anio'] = anio
    proveedores.append(temp)
proveedores = pd.concat(proveedores, ignore_index=True)
proveedores = proveedores.rename(columns={'main_ocid': 'ocid', 'name': 'proveedor_ganador'})

df['n_postores'] = pd.to_numeric(df['n_postores'], errors='coerce')
df['monto_adjudicado'] = pd.to_numeric(df['monto_adjudicado'], errors='coerce').fillna(0)
df['monto_MM'] = df['monto_adjudicado'] / 1_000_000
df['un_solo_postor'] = df['un_solo_postor'].astype(str).str.lower().eq('true')
df['fecha_proceso'] = pd.to_datetime(df['fecha_proceso'], errors='coerce')
df['mes'] = df['fecha_proceso'].dt.to_period('M').astype(str)
print(df.shape)


## Seccion 1. Resumen General

In [ ]:
total_procesos = df['ocid'].nunique()
print('KPI 1 - Total de procesos analizados:', f'{total_procesos:,}')


In [ ]:
pct_un_postor = df['un_solo_postor'].mean() * 100
print('KPI 2 - % de procesos con un solo postor:', f'{pct_un_postor:.2f}%')


In [ ]:
monto_total = df['monto_MM'].sum()
print('KPI 3 - Monto total adjudicado (MM PEN):', f'{monto_total:,.2f}')


In [ ]:
g1 = df.groupby('categoria', as_index=False).agg(monto_MM=('monto_MM', 'sum'))
fig = px.treemap(g1, path=['categoria'], values='monto_MM', title='Distribucion del monto adjudicado por categoria')
fig.show()


## Seccion 2. Competencia

In [ ]:
hist_df = df['n_postores'].dropna().clip(upper=10)
plt.figure(figsize=(10, 5))
plt.hist(hist_df, bins=range(0, 12), edgecolor='black', color='steelblue')
plt.title('Distribucion del numero de postores por proceso')
plt.xlabel('Numero de postores (10 = 10 o mas)')
plt.ylabel('Cantidad de procesos')
plt.tight_layout()
plt.show()


In [ ]:
g3 = (df[df['departamento'] != 'Sin dato'].groupby('departamento', as_index=False).agg(casos_un_postor=('un_solo_postor', 'sum')).sort_values('casos_un_postor', ascending=False).head(20))
plt.figure(figsize=(10, 7))
plt.barh(g3['departamento'], g3['casos_un_postor'], color='darkorange')
plt.title('Top 20 departamentos con un solo postor')
plt.xlabel('Cantidad de procesos')
plt.ylabel('Departamento')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
g4 = pd.crosstab(df['categoria'], df['metodo_simple'], normalize='index') * 100
g4.plot(kind='bar', stacked=True, figsize=(10, 5), colormap='Set2')
plt.title('Metodo de contratacion por categoria (%)')
plt.xlabel('Categoria')
plt.ylabel('Porcentaje')
plt.tight_layout()
plt.show()


In [ ]:
box_df = df[df['n_postores'].notna()].copy()
box_df['n_postores'] = box_df['n_postores'].clip(upper=15)
plt.figure(figsize=(10, 5))
sns.boxplot(data=box_df, x='categoria', y='n_postores', showfliers=False)
plt.title('Distribucion de postores por categoria')
plt.xlabel('Categoria')
plt.ylabel('Numero de postores')
plt.tight_layout()
plt.show()


## Seccion 3. Riesgo Economico

In [ ]:
g6 = df.groupby('anio', as_index=False).agg(monto_total=('monto_adjudicado', 'sum'), monto_riesgo=('monto_adjudicado', lambda s: s[df.loc[s.index, 'un_solo_postor']].sum()))
g6['monto_competitivo'] = g6['monto_total'] - g6['monto_riesgo']
g6['monto_riesgo_mm'] = g6['monto_riesgo'] / 1_000_000
g6['monto_competitivo_mm'] = g6['monto_competitivo'] / 1_000_000
x = np.arange(len(g6))
w = 0.35
plt.figure(figsize=(10, 5))
plt.bar(x - w/2, g6['monto_riesgo_mm'], width=w, label='Monto en riesgo')
plt.bar(x + w/2, g6['monto_competitivo_mm'], width=w, label='Monto competitivo')
plt.xticks(x, g6['anio'].astype(str))
plt.title('Monto en riesgo vs competitivo por anio')
plt.xlabel('Anio')
plt.ylabel('Millones de PEN')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
g7 = df.groupby(['anio', 'categoria'], as_index=False).agg(monto_MM=('monto_MM', 'sum'))
plt.figure(figsize=(10, 5))
sns.lineplot(data=g7, x='anio', y='monto_MM', hue='categoria', marker='o')
plt.title('Evolucion anual del gasto por categoria')
plt.xlabel('Anio')
plt.ylabel('Millones de PEN')
plt.tight_layout()
plt.show()


In [ ]:
g8 = (df[df['metodo_simple'] == 'Directa'].groupby('entidad_compradora', as_index=False).agg(n_directas=('ocid', 'nunique')).sort_values('n_directas', ascending=False).head(15))
plt.figure(figsize=(11, 7))
plt.barh(g8['entidad_compradora'], g8['n_directas'], color='forestgreen')
plt.title('Top 15 entidades con mas contrataciones directas')
plt.xlabel('Cantidad de directas')
plt.ylabel('Entidad compradora')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
g9 = df[(df['monto_adjudicado'] > 0) & (df['n_postores'].notna())].copy()
if len(g9) > 5000:
    g9 = g9.sample(5000, random_state=42)
plt.figure(figsize=(10, 5))
sns.scatterplot(data=g9, x='n_postores', y='monto_adjudicado', hue='un_solo_postor', alpha=0.5)
plt.title('Monto adjudicado vs numero de postores')
plt.xlabel('Numero de postores')
plt.ylabel('Monto adjudicado (PEN)')
plt.tight_layout()
plt.show()


## Seccion 4. Transparencia Geografica

In [ ]:
top_dptos = (df[df['departamento'] != 'Sin dato'].groupby('departamento')['ocid'].nunique().sort_values(ascending=False).head(15).index)
heat = (df[df['departamento'].isin(top_dptos)].groupby(['departamento', 'categoria'])['un_solo_postor'].mean().mul(100).reset_index().pivot(index='departamento', columns='categoria', values='un_solo_postor'))
plt.figure(figsize=(10, 7))
sns.heatmap(heat, annot=True, fmt='.1f', cmap='Reds')
plt.title('Riesgo por departamento y categoria (%)')
plt.tight_layout()
plt.show()


In [ ]:
g11 = (df[df['metodo_simple'].isin(['Competitivo', 'Directa'])].groupby(['mes', 'metodo_simple'], as_index=False).agg(procesos=('ocid', 'nunique')))
g11['mes_dt'] = pd.to_datetime(g11['mes'] + '-01')
g11 = g11.sort_values('mes_dt')
plt.figure(figsize=(12, 5))
sns.lineplot(data=g11, x='mes_dt', y='procesos', hue='metodo_simple', marker='o')
plt.title('Evolucion de contrataciones directas vs competitivas')
plt.xlabel('Mes')
plt.ylabel('Cantidad de procesos')
plt.tight_layout()
plt.show()


In [ ]:
g12 = (proveedores.groupby('proveedor_ganador', as_index=False).agg(procesos_ganados=('ocid', 'nunique')).sort_values('procesos_ganados', ascending=False).head(15))
plt.figure(figsize=(11, 7))
plt.barh(g12['proveedor_ganador'], g12['procesos_ganados'], color='firebrick')
plt.title('Top 15 proveedores ganadores repetidos')
plt.xlabel('Cantidad de procesos ganados')
plt.ylabel('Proveedor')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
score = (df[df['departamento'] != 'Sin dato'].groupby('departamento', as_index=False).agg(procesos=('ocid', 'nunique'), pct_directa=('metodo_simple', lambda s: (s == 'Directa').mean() * 100), pct_un_solo_postor=('un_solo_postor', 'mean'), monto_total=('monto_adjudicado', 'sum'), monto_riesgo=('monto_adjudicado', lambda s: s[df.loc[s.index, 'un_solo_postor']].sum())))
score['pct_un_solo_postor'] = score['pct_un_solo_postor'] * 100
score['pct_monto_riesgo'] = np.where(score['monto_total'] > 0, 100 * score['monto_riesgo'] / score['monto_total'], 0)
score['score_transparencia'] = 100 - (score['pct_directa'] + score['pct_un_solo_postor'] + score['pct_monto_riesgo']) / 3
score.sort_values('score_transparencia').head(15)[['departamento', 'procesos', 'pct_directa', 'pct_un_solo_postor', 'pct_monto_riesgo', 'score_transparencia']]
